# 10 — Long Context Architectures

Sliding window, sparse, and ring attention extend context windows beyond what standard attention supports.

## Sliding Window, Sparse, and Ring Attention

**Sliding window attention** (Longformer, Beltagy 2020): each token attends only to its *w* neighbours rather than all positions. Complexity O(n·w) — linear in sequence length. Global tokens (e.g., [CLS]) can attend to all positions, providing a bridge for long-range information. Used in GPT-NeoX and Mistral.

**Sparse attention** (BigBird, Zaheer 2020): combines sliding window + random attention + global tokens. The random connections ensure that any pair of tokens is connected within O(log n) hops (small world property), allowing O(n log n) information propagation.

**ALiBi / RoPE positional encodings**: instead of learned position embeddings, ALiBi adds a linear bias that decays with distance; RoPE encodes relative positions directly in the Q and K rotation. Both generalise better to sequence lengths longer than seen during training.

**Ring attention** (Liu et al., 2023): distributes long sequences across multiple GPUs by passing KV blocks in a ring, enabling trillion-token sequences.

**YaRN** (Peng et al., 2023): extends RoPE-based models to longer contexts via NTK interpolation, allowing e.g. Llama 2 (4k context) to handle 128k tokens with minimal fine-tuning.

In [ ]:
# Sliding window attention implementation
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class SlidingWindowAttention(nn.Module):
    def __init__(self, d_model, n_heads, window_size=8):
        super().__init__()
        self.d_head = d_model // n_heads
        self.n_heads = n_heads
        self.window_size = window_size
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, N, D = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.n_heads, self.d_head).permute(2,0,3,1,4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # (B, H, N, d)

        w = self.window_size
        outputs = []

        for i in range(N):
            # Window: [max(0, i-w), min(N, i+w+1)]
            lo = max(0, i - w)
            hi = min(N, i + w + 1)
            q_i = q[:, :, i:i+1, :]     # (B, H, 1, d)
            k_win = k[:, :, lo:hi, :]   # (B, H, win, d)
            v_win = v[:, :, lo:hi, :]
            attn = (q_i @ k_win.transpose(-2,-1) / self.d_head**0.5).softmax(dim=-1)
            out_i = (attn @ v_win).squeeze(2)  # (B, H, d)
            outputs.append(out_i)

        out = torch.stack(outputs, dim=2).transpose(1,2).reshape(B, N, D)
        return self.out(out)


torch.manual_seed(42)
sw_attn = SlidingWindowAttention(d_model=32, n_heads=4, window_size=4)
x = torch.randn(2, 64, 32)
out = sw_attn(x)
print(f'Sliding window output: {out.shape}')

# Verify it's O(n*w) not O(n^2)
w = 4
total_ops = 64 * (2*w+1)  # each position sees 2w+1 neighbours
std_ops = 64 * 64
print(f'Sliding window ops: {total_ops} vs standard: {std_ops} ({std_ops/total_ops:.1f}x reduction)')

In [ ]:
# Compare attention patterns: standard vs sliding window vs global
import matplotlib.pyplot as plt

N = 32

# Standard: full n x n
std_mask = torch.ones(N, N)

# Sliding window: w=4
sw_mask = torch.zeros(N, N)
for i in range(N):
    lo, hi = max(0, i-4), min(N, i+5)
    sw_mask[i, lo:hi] = 1

# BigBird: sliding window + 2 global + random
bb_mask = sw_mask.clone()
bb_mask[:, 0] = 1   # global token 0
bb_mask[:, -1] = 1  # global token n-1
# Random connections
torch.manual_seed(42)
for i in range(N):
    rand_idx = torch.randint(0, N, (2,))
    bb_mask[i, rand_idx] = 1

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, mask, title in zip(axes,
                             [std_mask, sw_mask, bb_mask],
                             ['Standard (n²)', 'Sliding Window (n·w)', 'BigBird (sliding+global+random)']):
    ax.imshow(mask, cmap='Blues', interpolation='nearest')
    ax.set_title(f'{title}\n(sparsity: {1-mask.mean():.1%})')
    ax.set_xlabel('Key position')
    ax.set_ylabel('Query position')

plt.suptitle('Long-Context Attention Patterns')
plt.tight_layout()
plt.savefig('/tmp/attention_patterns.png', dpi=100, bbox_inches='tight')
plt.show()
print('Attention pattern comparison saved')